<a href="https://colab.research.google.com/github/MarceloCoccaro12/Disciplina_de_Inteligencia_Artificial_UMC/blob/main/FISHMORPH_Family_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Dataset Utilizado
https://www.kaggle.com/datasets/menegidio/fishmorph-dataset

In [ ]:
!kaggle datasets download -d menegidio/fishmorph-dataset --unzip -p /content/dados

Dataset URL: https://www.kaggle.com/datasets/menegidio/fishmorph-dataset
License(s): MIT
100% 844k/844k [00:00<00:00, 105MB/s]



# Importando Bibliotecas

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
fish_db = pd.read_csv('/content/dados/FISHMORPH_Family_Dataset.csv')

In [ ]:
fish_db.head()

,MBl,BEl,VEp,REs,OGp,RMl,BLs,PFv,PFs,CPt,Family
0,130.0,4.579113,0.622642,0.375019,0.622642,0.868659,0.484981,0.273585,0.144108,3.348991,Cyprinidae
1,11.5,2.796068,0.650700,0.353780,0.509800,0.411157,0.722461,0.330169,0.221095,2.525592,Cichlidae
2,6.6,4.564329,0.518187,0.312002,0.203985,0.413221,0.591495,0.105517,0.188193,2.104580,Cyprinidae
3,10.9,4.390422,0.548247,0.244804,0.168959,0.324728,0.642991,0.097450,0.193049,1.787062,Cyprinidae
4,18.9,4.410271,0.542353,0.292058,0.183790,0.398567,0.685672,0.087643,0.170648,2.968906,Cyprinidae


# Valores duplicados

In [ ]:
fish_db.duplicated()

,0
0,False
1,False
2,False
3,False
4,False
...,...
8337,False
8338,False
8339,False
8340,False


# Valores Nulos

In [ ]:
fish_db.isnull().sum()

,0
MBl,0
BEl,0
VEp,0
REs,0
OGp,0
RMl,0
BLs,0
PFv,0
PFs,0
CPt,0


# Descrição das colunas

In [ ]:
fish_db.describe()

,MBl,BEl,VEp,REs,OGp,RMl,BLs,PFv,PFs,CPt
count,8342.000000,8342.000000,8342.000000,8342.000000,8342.000000,8342.000000,8342.000000,8342.000000,8342.000000,8342.000000
mean,22.004517,4.452438,0.549087,0.391665,0.381768,0.399786,0.571818,0.285316,0.189343,2.560795
std,33.612007,2.338235,0.094883,0.132814,0.194352,0.302658,0.117486,0.155629,0.056109,1.016659
min,1.200000,1.033084,0.159248,0.000000,0.000000,0.000000,0.224525,0.000000,0.000000,0.000000
25%,7.000000,3.139102,0.487993,0.293205,0.278014,0.257999,0.490052,0.193134,0.157972,1.999744
50%,12.600000,4.038851,0.546213,0.389429,0.406904,0.361436,0.564739,0.275039,0.183497,2.436329
75%,24.500000,5.002621,0.605125,0.486221,0.516001,0.490671,0.648039,0.368876,0.215031,2.919818
max,800.000000,43.728161,1.000000,1.026667,1.026316,7.060344,1.230105,0.857097,0.511113,15.452061


# Informações sobre os tipos de variaveis


In [ ]:
fish_db.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8342 entries, 0 to 8341
Data columns (total 11 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   MBl     8342 non-null   float64
 1   BEl     8342 non-null   float64
 2   VEp     8342 non-null   float64
 3   REs     8342 non-null   float64
 4   OGp     8342 non-null   float64
 5   RMl     8342 non-null   float64
 6   BLs     8342 non-null   float64
 7   PFv     8342 non-null   float64
 8   PFs     8342 non-null   float64
 9   CPt     8342 non-null   float64
 10  Family  8342 non-null   object 
dtypes: float64(10), object(1)
memory usage: 717.0+ KB


# Verificando quantos valores nao repetidos na classe (family)

In [ ]:
print(fish_db['Family'].unique())

['Cyprinidae' 'Cichlidae' 'Anostomidae' 'Abyssocottidae' 'Centrarchidae'
 'Loricariidae' 'Characidae' 'Mochokidae' 'Nemacheilidae' 'Doradidae'
 'Gobiidae' 'Sparidae' 'Trichomycteridae' 'Cobitidae' 'Heptapteridae'
 'Acestrorhynchidae' 'Soleidae' 'Achiridae' 'Acipenseridae' 'Akysidae'
 'Apteronotidae' 'Nandidae' 'Auchenipteridae' 'Mugilidae' 'Pimelodidae'
 'Schilbeidae' 'Alestidae' 'Poeciliidae' 'Goodeidae' 'Eleotridae'
 'Clupeidae' 'Aspredinidae' 'Bagridae' 'Engraulidae' 'Ambassidae'
 'Amblycipitidae' 'Ictaluridae' 'Amiidae' 'Percidae' 'Crenuchidae'
 'Terapontidae' 'Amphiliidae' 'Anabantidae' 'Anablepidae' 'Rivulidae'
 'Ariidae' 'Anguillidae' 'Balitoridae' 'Plotosidae' 'Hemiodontidae'
 'Parodontidae' 'Gasterosteidae' 'Cyprinodontidae' 'Aphredoderidae'
 'Aplocheilidae' 'Galaxiidae' 'Sciaenidae' 'Arapaimidae' 'Sternopygidae'
 'Gobiesocidae' 'Hemiramphidae' 'Callichthyidae' 'Astroblepidae'
 'Atherinidae' 'Atherinopsidae' 'Lepisosteidae' 'Tetraodontidae'
 'Austroglanididae' 'Badidae' 'Sisor

In [ ]:
print("Contagem de repetições por familia :\n", fish_db['Family'].value_counts())

Contagem de repetições por familia :
 Family
Cyprinidae         1545
Cichlidae          1060
Characidae          732
Loricariidae        427
Percidae            190
                   ... 
Sillaginidae          1
Serrasalmidae         1
Bythitidae            1
Vaillantellidae       1
Valenciidae           1
Name: count, Length: 197, dtype: int64


# Removendo da classe Familias no qual aparecem menos de 5 vezes

In [ ]:
contagem = fish_db['Family'].value_counts()

familias_validas = contagem[contagem >= 5].index

fish_db = fish_db[fish_db['Family'].isin(familias_validas)]

In [ ]:
X = fish_db.drop('Family', axis=1)
y = fish_db['Family']

In [ ]:
numerical_features = [
    'MBl',
    'BEl',
    'VEp',
    'REs',
    'OGp',
    'RMl',
    'BLs',
    'PFv',
    'PFs',
    'CPt'
]

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features)
    ],
    remainder='passthrough'
)

In [ ]:
model_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                        ('classifier', SVC(kernel='linear'))])

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y , test_size=0.2, random_state=42)

In [ ]:
model_pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('num', StandardScaler(),
                                                  ['MBl', 'BEl', 'VEp', 'REs',
                                                   'OGp', 'RMl', 'BLs', 'PFv',
                                                   'PFs', 'CPt'])])),
                ('classifier', SVC(kernel='linear'))])

In [ ]:
y_pred = model_pipeline.predict(X_test)

In [ ]:
acuracia = accuracy_score(y_test, y_pred)
print(f'Acurácia kernel linear: {acuracia * 100:.2f}%')

Acurácia kernel linear: 64.49%


# Utilizando o GridSearchCV

In [ ]:
param_grid = {
    'classifier__C': [0.1, 1, 10, 100],
    'classifier__gamma': ['scale', 'auto', 0.01, 0.1, 1],
    'classifier__kernel': ['linear', 'poly', 'rbf', 'sigmoid']
}

grid_search = GridSearchCV(
    model_pipeline,
    param_grid,
    cv = 5,
    scoring='accuracy',
    verbose = 2,
    n_jobs = -1
)

grid_search.fit(X_train, y_train)

# Obter melhores parâmetros e melhor pontuação de validação cruzada
best_param = grid_search.best_params_
best_score = grid_search.best_score_

print('\n ----Melhores Hiperparâmetros encontrados (Validação Cruzada)----')
print(f'Melhores parametros encontrados: {best_param}')
print(f'Melhor Acurácia Média (Validação Cruzada): {best_score}')

Fitting 5 folds for each of 80 candidates, totalling 400 fits


/usr/local/lib/python3.13/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 3 members, which is less than n_splits=5.
  warnings.warn(



 ----Melhores Hiperparâmetros encontrados (Validação Cruzada)----
Melhores parametros encontrados: {'classifier__C': 10, 'classifier__gamma': 'scale', 'classifier__kernel': 'rbf'}
Melhor Acurácia Média (Validação Cruzada): 0.6609695995714476


# Avaliar o melhor modelo do conjunto de teste


In [ ]:
best_model = grid_search.best_estimator_
y_pred_tuned = best_model.predict(X_test)
acuracia_teste = accuracy_score(y_test, y_pred_tuned)
print(f'Acuracia teste: {acuracia_teste * 100:.2f}%')

Acuracia teste: 68.94%


In [ ]:
print(classification_report(y_test, y_pred_tuned))

                   precision    recall  f1-score   support

   Abyssocottidae       1.00      0.67      0.80         3
Acestrorhynchidae       0.57      1.00      0.73         4
        Achiridae       1.00      0.25      0.40         4
    Acipenseridae       1.00      0.75      0.86         4
 Adrianichthyidae       0.00      0.00      0.00         1
         Akysidae       0.33      1.00      0.50         1
        Alestidae       0.00      0.00      0.00        15
       Ambassidae       0.50      1.00      0.67         1
   Amblycipitidae       0.25      0.50      0.33         2
      Amphiliidae       0.86      0.46      0.60        13
      Anabantidae       0.50      0.33      0.40         3
      Anablepidae       0.00      0.00      0.00         4
      Anguillidae       1.00      1.00      1.00         2
      Anostomidae       0.47      0.33      0.39        24
    Aplocheilidae       0.38      0.50      0.43        12
    Apteronotidae       0.60      0.50      0.55       

/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_

# Matriz de Confusão

In [ ]:
cm = confusion_matrix(y_test, y_pred_tuned)
print(cm)

[[2 0 0 ... 0 0 0]
 [0 4 0 ... 0 0 0]
 [0 0 1 ... 0 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 7 0]
 [0 0 0 ... 0 0 0]]
